# Automation Of Python Code In Snowflake!

This is a notebook to demo the ways to automate the python code in Snowflake. In the training part we will create a simple model and push it to registry. The automation is done on the inference code of this model. Happy learning ;)

In [ ]:
%%sql -r dataframe_1
-- SQL Cell
CREATE DATABASE IF NOT EXISTS ML_LAB;
CREATE SCHEMA IF NOT EXISTS ML_LAB.DATA;

CREATE OR REPLACE TABLE ML_LAB.DATA.IRIS (
    sepal_length FLOAT,
    sepal_width  FLOAT,
    petal_length FLOAT,
    petal_width  FLOAT,
    species      VARCHAR
);

In [ ]:
# Python Cell
from snowflake.snowpark.context import get_active_session
from sklearn.datasets import load_iris

session = get_active_session()

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.columns = ['SEPAL_LENGTH', 'SEPAL_WIDTH', 'PETAL_LENGTH', 'PETAL_WIDTH', 'SPECIES']
df['SPECIES'] = df['SPECIES'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

session.write_pandas(df, "IRIS", database="ML_LAB", schema="DATA", overwrite=True)
print(f"Loaded {len(df)} rows into ML_LAB.DATA.IRIS")

In [ ]:
# Python Cell
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from snowflake.ml.registry import Registry

df = session.table("ML_LAB.DATA.IRIS").to_pandas()

X = df[['SEPAL_LENGTH', 'SEPAL_WIDTH', 'PETAL_LENGTH', 'PETAL_WIDTH']]
y = df['SPECIES']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy: {accuracy:.2%}")

reg = Registry(session=session, database_name="ML_LAB", schema_name="DATA")
reg.log_model(
    model=model,
    model_name="iris_classifier",
    version_name="v1",
    sample_input_data=X_train,
    metrics={"accuracy": accuracy},
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    comment="Random Forest trained on Iris dataset"
)
print("Model pushed to registry: iris_classifier / v1")

### Done! Your model is now in the Snowflake Model Registry. Everyone on your team can load it by name — no training code, no model files to share. This is the only time you run the training notebook.